In [1]:
import pandas as pd

df_data = pd.read_csv("../data/raw/EdStatsData.csv")
print("Shape:", df_data.shape)
df_data.head()

Shape: (886930, 70)


,Country Name,Country Code,Indicator Name,Indicator Code,1970,1971,1972,1973,1974,1975,...,2060,2065,2070,2075,2080,2085,2090,2095,2100,Unnamed: 69
0,Arab World,ARB,"Adjusted net enrolment rate, lower secondary, ...",UIS.NERA.2,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Arab World,ARB,"Adjusted net enrolment rate, lower secondary, ...",UIS.NERA.2.F,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Arab World,ARB,"Adjusted net enrolment rate, lower secondary, ...",UIS.NERA.2.GPI,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Arab World,ARB,"Adjusted net enrolment rate, lower secondary, ...",UIS.NERA.2.M,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Arab World,ARB,"Adjusted net enrolment rate, primary, both sex...",SE.PRM.TENR,54.822121,54.894138,56.209438,57.267109,57.991138,59.36554,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [2]:
print("Países únicos:", df_data["Country Name"].nunique())
print("Indicadores únicos:", df_data["Indicator Name"].nunique())
print()
print("Valores ausentes por coluna (10 primeiras):")
print(df_data.isnull().sum().head(10))

Países únicos: 242
Indicadores únicos: 3665

Valores ausentes por coluna (10 primeiras):
Country Name           0
Country Code           0
Indicator Name         0
Indicator Code         0
1970              814642
1971              851393
1972              851311
1973              851385
1974              851200
1975              799624
dtype: int64


In [3]:
df_country = pd.read_csv("../data/raw/EdStatsCountry.csv")
df_series = pd.read_csv("../data/raw/EdStatsSeries.csv")

print("Colunas de EdStatsCountry:")
print(df_country.columns.tolist())
print()
print("Exemplo de grupos de renda (Income Group):")
print(df_country["Income Group"].unique())

Colunas de EdStatsCountry:
['Country Code', 'Short Name', 'Table Name', 'Long Name', '2-alpha code', 'Currency Unit', 'Special Notes', 'Region', 'Income Group', 'WB-2 code', 'National accounts base year', 'National accounts reference year', 'SNA price valuation', 'Lending category', 'Other groups', 'System of National Accounts', 'Alternative conversion factor', 'PPP survey year', 'Balance of Payments Manual in use', 'External debt Reporting status', 'System of trade', 'Government Accounting concept', 'IMF data dissemination standard', 'Latest population census', 'Latest household survey', 'Source of most recent Income and expenditure data', 'Vital registration complete', 'Latest agricultural census', 'Latest industrial data', 'Latest trade data', 'Latest water withdrawal data', 'Unnamed: 31']

Exemplo de grupos de renda (Income Group):
<StringArray>
['High income: nonOECD',           'Low income',  'Upper middle income',
                    nan,  'Lower middle income',    'High income:

In [4]:
# quantos países "de verdade" sobram, filtrando os que tem Income Group vazio
paises_de_verdade = df_country[df_country["Income Group"].notna()]
print("Total de linhas no EdStatsCountry:", df_country.shape[0])
print("Países 'de verdade' (com Income Group preenchido):", paises_de_verdade.shape[0])

# espiando alguns exemplos dos que SÃO agrupamentos (Income Group vazio)
agrupamentos = df_country[df_country["Income Group"].isna()]
print()
print("Exemplos de agrupamentos (não são países):")
print(agrupamentos["Short Name"].head(10).tolist())

Total de linhas no EdStatsCountry: 241
Países 'de verdade' (com Income Group preenchido): 214

Exemplos de agrupamentos (não são países):
['Arab World', 'East Asia & Pacific (developing only)', 'East Asia & Pacific (all income levels)', 'Europe & Central Asia (developing only)', 'Europe & Central Asia (all income levels)', 'Euro area', 'European Union', 'Gibraltar', 'High income', 'Heavily indebted poor countries (HIPC)']


In [5]:
# procurando indicadores interessantes pelo nome
termos_interesse = "literacy|expenditure|enrol|teacher|PISA|primary completion"

indicadores_candidatos = df_series[
    df_series["Indicator Name"].str.contains(termos_interesse, case=False, na=False)
]

print("Total de indicadores encontrados:", indicadores_candidatos.shape[0])
indicadores_candidatos[["Series Code", "Indicator Name"]].head(30)

Total de indicadores encontrados: 721


,Series Code,Indicator Name
440,HH.DHS.PCR,DHS: Primary completion rate
441,HH.DHS.PCR.F,DHS: Primary completion rate. Female
442,HH.DHS.PCR.M,DHS: Primary completion rate. Male
443,HH.DHS.PCR.Q1,DHS: Primary completion rate. Quintile 1
444,HH.DHS.PCR.Q2,DHS: Primary completion rate. Quintile 2
445,HH.DHS.PCR.Q3,DHS: Primary completion rate. Quintile 3
446,HH.DHS.PCR.Q4,DHS: Primary completion rate. Quintile 4
447,HH.DHS.PCR.Q5,DHS: Primary completion rate. Quintile 5
448,HH.DHS.PCR.R,DHS: Primary completion rate. Rural
449,HH.DHS.PCR.U,DHS: Primary completion rate. Urban


In [6]:
indicadores_escolhidos = [
    "SE.ADT.LITR.ZS",
    "SE.XPD.TOTL.GD.ZS",
    "SE.PRM.ENRR",
    "SE.SEC.ENRR",
    "SE.PRM.CMPT.ZS",
    "SE.PRM.TCHR",
    "SE.LPV.PRIM",
    "SE.XPD.PRIM.ZS",
]

conferencia = df_series[df_series["Series Code"].isin(indicadores_escolhidos)]
conferencia[["Series Code", "Indicator Name"]]

,Series Code,Indicator Name
2215,SE.ADT.LITR.ZS,"Adult literacy rate, population 15+ years, bot..."
2241,SE.PRM.CMPT.ZS,"Primary completion rate, both sexes (%)"
2250,SE.PRM.ENRR,"Gross enrolment ratio, primary, both sexes (%)"
2275,SE.PRM.TCHR,"Teachers in primary education, both sexes (num..."
2307,SE.SEC.ENRR,"Gross enrolment ratio, secondary, both sexes (%)"
2375,SE.XPD.PRIM.ZS,Expenditure on primary as % of government expe...
2381,SE.XPD.TOTL.GD.ZS,Government expenditure on education as % of GD...


In [7]:
pisa_leitura = df_series[
    df_series["Indicator Name"].str.contains("PISA|reading", case=False, na=False)
]
pisa_leitura[["Series Code", "Indicator Name"]].head(20)

,Series Code,Indicator Name
695,LO.EGRA.CWPM.ZERO.AFA.2GRD,EGRA: Oral Reading Fluency - Share of students...
696,LO.EGRA.CWPM.ZERO.AFA.3GRD,EGRA: Oral Reading Fluency - Share of students...
697,LO.EGRA.CWPM.ZERO.AKU.2GRD,EGRA: Oral Reading Fluency - Share of students...
698,LO.EGRA.CWPM.ZERO.AMH.2GRD,EGRA: Oral Reading Fluency - Share of students...
699,LO.EGRA.CWPM.ZERO.AMH.3GRD,EGRA: Oral Reading Fluency - Share of students...
700,LO.EGRA.CWPM.ZERO.ARB.2GRD,EGRA: Oral Reading Fluency - Share of students...
701,LO.EGRA.CWPM.ZERO.ARB.3GRD,EGRA: Oral Reading Fluency - Share of students...
702,LO.EGRA.CWPM.ZERO.AST.2GRD,EGRA: Oral Reading Fluency - Share of students...
703,LO.EGRA.CWPM.ZERO.BMN.2GRD,EGRA: Oral Reading Fluency - Share of students...
704,LO.EGRA.CWPM.ZERO.BOM.2GRD,EGRA: Oral Reading Fluency - Share of students...


In [8]:
# lista final (sem o PISA)
indicadores_escolhidos = [
    "SE.ADT.LITR.ZS",
    "SE.XPD.TOTL.GD.ZS",
    "SE.PRM.ENRR",
    "SE.SEC.ENRR",
    "SE.PRM.CMPT.ZS",
    "SE.PRM.TCHR",
    "SE.XPD.PRIM.ZS",
]

# lista de países "de verdade" (Income Group preenchido), que descobrimos ontem
paises_validos = df_country[df_country["Income Group"].notna()]["Country Code"].tolist()

# filtrando o dataset principal: só os indicadores escolhidos + só países de verdade
df_filtrado = df_data[
    (df_data["Indicator Code"].isin(indicadores_escolhidos)) &
    (df_data["Country Code"].isin(paises_validos))
]

print("Shape original:", df_data.shape)
print("Shape filtrado:", df_filtrado.shape)
df_filtrado.head()

Shape original: (886930, 70)
Shape filtrado: (1498, 70)


,Country Name,Country Code,Indicator Name,Indicator Code,1970,1971,1972,1973,1974,1975,...,2060,2065,2070,2075,2080,2085,2090,2095,2100,Unnamed: 69
91645,Afghanistan,AFG,"Adult literacy rate, population 15+ years, bot...",SE.ADT.LITR.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
92857,Afghanistan,AFG,Expenditure on primary as % of government expe...,SE.XPD.PRIM.ZS,NaN,26.537170,NaN,NaN,NaN,36.814362,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
92885,Afghanistan,AFG,Government expenditure on education as % of GD...,SE.XPD.TOTL.GD.ZS,NaN,1.160360,1.117180,1.427880,NaN,1.303320,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
92956,Afghanistan,AFG,"Gross enrolment ratio, primary, both sexes (%)",SE.PRM.ENRR,31.341749,32.210339,32.507469,32.873798,33.437599,33.316231,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
92960,Afghanistan,AFG,"Gross enrolment ratio, secondary, both sexes (%)",SE.SEC.ENRR,8.331610,9.350290,10.348610,10.831690,10.976400,11.041030,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
# pegando só as colunas de ano que interessam (vamos focar em 1990-2020, dados mais completos)
colunas_ano = [str(ano) for ano in range(1990, 2021)]

# "deitando" a tabela: uma linha por país + indicador + ano
df_long = df_filtrado.melt(
    id_vars=["Country Name", "Country Code", "Indicator Name", "Indicator Code"],
    value_vars=colunas_ano,
    var_name="Ano",
    value_name="Valor"
)

df_long["Ano"] = df_long["Ano"].astype(int)

print("Shape depois do melt:", df_long.shape)
df_long.head(10)

KeyError: "The following id_vars or value_vars are not present in the DataFrame: ['2018', '2019']"

In [10]:
print(df_filtrado.columns.tolist())

['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code', '1970', '1971', '1972', '1973', '1974', '1975', '1976', '1977', '1978', '1979', '1980', '1981', '1982', '1983', '1984', '1985', '1986', '1987', '1988', '1989', '1990', '1991', '1992', '1993', '1994', '1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2020', '2025', '2030', '2035', '2040', '2045', '2050', '2055', '2060', '2065', '2070', '2075', '2080', '2085', '2090', '2095', '2100', 'Unnamed: 69']


In [11]:
# ajustando: dados anuais reais vão de 1990 até 2017 (sem buracos)
colunas_ano = [str(ano) for ano in range(1990, 2018)]

df_long = df_filtrado.melt(
    id_vars=["Country Name", "Country Code", "Indicator Name", "Indicator Code"],
    value_vars=colunas_ano,
    var_name="Ano",
    value_name="Valor"
)

df_long["Ano"] = df_long["Ano"].astype(int)

print("Shape depois do melt:", df_long.shape)
df_long.head(10)

Shape depois do melt: (41944, 6)


,Country Name,Country Code,Indicator Name,Indicator Code,Ano,Valor
0,Afghanistan,AFG,"Adult literacy rate, population 15+ years, bot...",SE.ADT.LITR.ZS,1990,NaN
1,Afghanistan,AFG,Expenditure on primary as % of government expe...,SE.XPD.PRIM.ZS,1990,NaN
2,Afghanistan,AFG,Government expenditure on education as % of GD...,SE.XPD.TOTL.GD.ZS,1990,NaN
3,Afghanistan,AFG,"Gross enrolment ratio, primary, both sexes (%)",SE.PRM.ENRR,1990,30.09115
4,Afghanistan,AFG,"Gross enrolment ratio, secondary, both sexes (%)",SE.SEC.ENRR,1990,11.22283
5,Afghanistan,AFG,"Primary completion rate, both sexes (%)",SE.PRM.CMPT.ZS,1990,NaN
6,Afghanistan,AFG,"Teachers in primary education, both sexes (num...",SE.PRM.TCHR,1990,15106.00000
7,Albania,ALB,"Adult literacy rate, population 15+ years, bot...",SE.ADT.LITR.ZS,1990,NaN
8,Albania,ALB,Expenditure on primary as % of government expe...,SE.XPD.PRIM.ZS,1990,NaN
9,Albania,ALB,Government expenditure on education as % of GD...,SE.XPD.TOTL.GD.ZS,1990,NaN


In [12]:
total_linhas = df_long.shape[0]
total_nulos = df_long["Valor"].isnull().sum()

print(f"Total de linhas: {total_linhas}")
print(f"Valores ausentes: {total_nulos} ({total_nulos/total_linhas:.1%})")
print()

# ausência por indicador (quais indicadores têm mais buracos?)
ausencia_por_indicador = df_long.groupby("Indicator Name")["Valor"].apply(
    lambda x: x.isnull().mean()
).sort_values(ascending=False)

print("Percentual de dados ausentes por indicador:")
print((ausencia_por_indicador * 100).round(1))

Total de linhas: 41944
Valores ausentes: 23117 (55.1%)

Percentual de dados ausentes por indicador:
Indicator Name
Adult literacy rate, population 15+ years, both sexes (%)                 89.6
Expenditure on primary as % of government expenditure on education (%)    69.5
Government expenditure on education as % of GDP (%)                       58.3
Primary completion rate, both sexes (%)                                   51.4
Teachers in primary education, both sexes (number)                        43.2
Gross enrolment ratio, secondary, both sexes (%)                          42.1
Gross enrolment ratio, primary, both sexes (%)                            31.6
Name: Valor, dtype: float64


In [13]:
# 1. Remover o indicador de alfabetização (cobertura ruim demais)
df_long = df_long[df_long["Indicator Code"] != "SE.ADT.LITR.ZS"]

# 2. Preencher buracos "olhando pro país", usando o valor mais próximo no tempo
df_long = df_long.sort_values(["Country Code", "Indicator Code", "Ano"])

df_long["Valor"] = df_long.groupby(["Country Code", "Indicator Code"])["Valor"].transform(
    lambda x: x.ffill().bfill()
)

# conferindo o resultado
total_linhas = df_long.shape[0]
total_nulos = df_long["Valor"].isnull().sum()
print(f"Total de linhas: {total_linhas}")
print(f"Valores ausentes depois do tratamento: {total_nulos} ({total_nulos/total_linhas:.1%})")

Total de linhas: 35952
Valores ausentes depois do tratamento: 4116 (11.4%)


In [14]:
df_long.to_csv("../data/processed/edstats_tratado.csv", index=False)
print("Arquivo salvo com sucesso!")
print("Shape final:", df_long.shape)

Arquivo salvo com sucesso!
Shape final: (35952, 6)
